# Fine-Tune Gemma 4 26B-A4B for Legal Partner

Fine-tunes `google/gemma-4-26B-A4B-it` (MoE: 26B total, 4B active) on 1,827 legal task examples using QLoRA.

**Run on:** RunPod A100 80GB (~$1.50/hr) or any machine with A100 80GB + 200GB disk.
**Training time:** ~10-16 hours (~$20 total).

**Tasks trained:** Drafting, Risk Assessment, Extraction, Checklist, Redline

---

### RunPod Setup
1. Go to [runpod.io](https://runpod.io)
2. Deploy → GPU Cloud → **A100 80GB SXM** or **A100 80GB PCIe**
3. Template: **RunPod PyTorch 2.4** (or any CUDA 12+ image)
4. Disk: **200GB** container disk + 50GB volume
5. Start pod → Open Jupyter → Upload this notebook

In [ ]:
# Cell 1 - Check GPU (must show A100 80GB)
!nvidia-smi
!df -h / | head -2

In [ ]:
# Cell 2 - Install dependencies
!pip install git+https://github.com/huggingface/transformers.git
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

In [ ]:
# Cell 3 - HuggingFace login
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")  # https://huggingface.co/settings/tokens

In [ ]:
# Cell 4 - Clone training data
!git clone https://github.com/jyoti0512shukla/legal-finetune.git /content/legal-finetune 2>/dev/null || echo 'Already cloned'
!cd /content/legal-finetune && git checkout v3
!wc -l /content/legal-finetune/data/gemma4/train.jsonl
!wc -l /content/legal-finetune/data/gemma4/validation.jsonl

In [ ]:
# Cell 5 - Load Gemma 4 26B-A4B with LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "google/gemma-4-26B-A4B-it",
    max_seq_length = 4096,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 128,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

model.print_trainable_parameters()

In [ ]:
# Cell 6 - Load and format training data
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma")

dataset = load_dataset("json",
    data_files="/content/legal-finetune/data/gemma4/train.jsonl",
    split="train")

def format_chat(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False
            )
            for c in examples["conversations"]
        ]
    }

train_data = dataset.map(format_chat, batched=True)
print(f"Training examples: {len(train_data)}")
print(f"\nSample:\n{train_data[0]['text'][:500]}")

In [ ]:
# Cell 7 - Train (~10-16 hours)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_data,
    dataset_text_field = "text",
    max_seq_length     = 4096,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 8,
        num_train_epochs             = 3,
        learning_rate                = 1e-4,
        bf16                         = True,
        logging_steps                = 25,
        save_steps                   = 100,
        save_total_limit             = 3,
        optim                        = "adamw_8bit",
        lr_scheduler_type            = "cosine",
        warmup_steps                 = 17,
        weight_decay                 = 0.01,
        output_dir                   = "/workspace/gemma4-legal-checkpoints",
    ),
)

trainer_stats = trainer.train()
print(f"\nDone. Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Cell 8 - Validate
FastLanguageModel.for_inference(model)

prompts = [
    ("DRAFT", "Draft a termination clause for a vendor agreement governed by Indian law. Write exactly 4 numbered sub-clauses as plain legal prose."),
    ("RISK", "Assess the risk level of this clause:\n\n'The vendor shall not be liable for any damages whatsoever arising from this agreement, whether direct, indirect, or consequential.'"),
    ("EXTRACT", "Extract key terms from this clause:\n\n'This Agreement between Acme Pvt Ltd (Vendor) and Beta Corp (Client), effective 1 January 2026, for a term of 2 years at INR 24,00,000 per annum, governed by the laws of India, with arbitration in Mumbai.'"),
]

for label, prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    result = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}\n{label}\n{'='*60}\n{result[:600]}\n")

# CHECK:
# DRAFT   -> plain prose numbered clauses, NO JSON
# RISK    -> structured risk output with level + issues
# EXTRACT -> labelled key terms

In [ ]:
# Cell 9 - Save adapter to HuggingFace
model.push_to_hub("jyoti0512shuklaorg/gemma4-legal-v1", private=True)
tokenizer.push_to_hub("jyoti0512shuklaorg/gemma4-legal-v1", private=True)
print("Adapter saved")

In [ ]:
# Cell 10 - Merge into full model + push to HuggingFace
model.save_pretrained_merged(
    "/workspace/gemma4-legal-v1-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved locally")

model.push_to_hub_merged(
    "jyoti0512shuklaorg/gemma4-legal-v1-merged",
    tokenizer,
    save_method="merged_16bit",
    private=True,
)
print("Merged model pushed to HuggingFace")

In [ ]:
# Cell 11 - Export GGUF for customer L4 VMs
model.push_to_hub_gguf(
    "jyoti0512shuklaorg/gemma4-legal-v1-gguf",
    tokenizer,
    quantization_method="q4_k_m",
    private=True,
)
print("GGUF pushed to HuggingFace")

## After Training - Deploy

### Serve on Colab A100 40GB (testing)
```bash
# Pull the 4-bit quantized model (~15GB) - NOT the full 100GB
vllm serve jyoti0512shuklaorg/gemma4-legal-v1-merged \
  --port 8000 --host 0.0.0.0 \
  --max-model-len 8192 \
  --quantization awq \
  --gpu-memory-utilization 0.90 \
  --max-num-seqs 5
```

### Serve on customer L4 VMs (production)
```bash
# Download GGUF (~15GB)
huggingface-cli download jyoti0512shuklaorg/gemma4-legal-v1-gguf --local-dir ./model

# Serve with llama.cpp
llama-server -m ./model/gemma4-legal-v1-q4_k_m.gguf \
  --ctx-size 8192 --port 8000 -np 5
```

### Backend .env
```
LEGALPARTNER_CHAT_PROVIDER=vllm
LEGALPARTNER_CHAT_API_URL=http://localhost:8000/v1
LEGALPARTNER_CHAT_API_MODEL=jyoti0512shuklaorg/gemma4-legal-v1-merged
```

### Shut down RunPod after training to stop billing!